<a href="https://colab.research.google.com/github/Deba088/DeepSurfaceClassifier/blob/implement-train-generator/notebook/train_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Module

In [37]:
!pip install logger

In [38]:
import logging
import os
import shutil
import random
from datetime import datetime

import cv2
import numpy as np
import tensorflow as tf
from logger import logger
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import numpy as np


# GPU Configuration

# Train and Test Dataset Generate
Create a generator which will split and generate train and test dataset

In [39]:
__all__ = ["train_generator", "test_generator", "train_data_length", "test_data_length"]

tf.get_logger().setLevel(logging.ERROR)

## Data Ingestion from GDrive

In [40]:
# Mount drive
from google.colab import drive
drive.mount("/content/drive")

# Set train data directory
GDRIVE_DATASET_DIR = "/content/drive/MyDrive/Google Colab Datasets/rscd_dataset"
logger.info(f"Train data location: {GDRIVE_DATASET_DIR}/train_1000")
logger.info(f"Model weight location: {GDRIVE_DATASET_DIR}/model_weights")

Mounted at /content/drive
[10/Sep/2023 13:42:36] INFO - Train data location: /content/drive/MyDrive/Google Colab Datasets/rscd_dataset/train_1000
[10/Sep/2023 13:42:36] INFO - Model weight location: /content/drive/MyDrive/Google Colab Datasets/rscd_dataset/model_weights


In [41]:
# Source folder you want to copy
source_folder = f"{GDRIVE_DATASET_DIR}/train_1000"

# Destination folder where you want to copy the contents
destination_folder = "/content/train_1000"

DATASET_DIR = destination_folder

if os.path.exists(DATASET_DIR):
  logger.info(f"Path already exists in current session")
else:
  logger.info(f"Path does not exists in current session")

  # Copy the entire folder and its contents from the source to the destination
  shutil.copytree(source_folder, destination_folder)

  logger.info(f"All files copied from {source_folder} to {destination_folder}")

[10/Sep/2023 13:42:36] INFO - Path does not exists in current session
[10/Sep/2023 13:51:08] INFO - All files copied from /content/drive/MyDrive/Google Colab Datasets/rscd_dataset/train_1000 to /content/train_1000


## Define Classes

In [42]:
# Get list of class names
CLASS_NAMES = sorted(os.listdir(DATASET_DIR))
CLASS_NAMES = ["dry_asphalt_severe", "dry_gravel", "fresh_snow", "water_asphalt_severe",
               "wet_asphalt_severe"]

# Log number of classes and class names
NUM_CLASSES = len(CLASS_NAMES)
logger.info(f"Number of classes: {NUM_CLASSES}")
logger.info(f"Class names: {CLASS_NAMES}")

[10/Sep/2023 13:51:08] INFO - Number of classes: 5
[10/Sep/2023 13:51:08] INFO - Class names: ['dry_asphalt_severe', 'dry_gravel', 'fresh_snow', 'water_asphalt_severe', 'wet_asphalt_severe']


## Define Image Dimension, Batch Size, Test Data Size and No of Epoch

In [43]:
# Define constants
IMG_HEIGHT, IMG_WEIGHT = 299, 299
BATCH_SIZE = 64
TEST_SIZE = 0.3
EPOCH = 10

## Split Train and Test Data

In [44]:
# Store list of images as (class_name, image_name)
IMG_LIST = []

for class_name in CLASS_NAMES:
  img_dir = os.path.join(DATASET_DIR, class_name)
  for img in os.listdir(img_dir):
    IMG_LIST.append((img, class_name))

IMG_LIST = np.array(IMG_LIST)

# Shuffle the list of images
np.random.shuffle(IMG_LIST)

In [45]:
# Split into train and test sets
train_data, test_data = [], []
for i in IMG_LIST:
  if random.random() < TEST_SIZE:
    test_data.append(i)
  else:
    train_data.append(i)

train_data = np.array(train_data)
test_data = np.array(test_data)

train_data_length = len(train_data)
test_data_length = len(test_data)

print(f"No of total data: {len(IMG_LIST)}")
print(f"No of train data: {train_data_length}")
print(f"No of test data: {test_data_length}")

No of total data: 5000
No of train data: 3551
No of test data: 1449


## Preprocess Image Data and Create Generator Object

In [46]:
# Preprocess image
def preprocess_image(frame, img_height,img_weight):
  frame = cv2.resize(frame, (img_height, img_weight))
  frame = img_to_array(frame)
  frame = frame / 255.0
  return frame

In [47]:
# Load data into train and test data in the required format
def data_generator(data, batch_size=BATCH_SIZE, img_height=IMG_HEIGHT,img_weight=IMG_WEIGHT):
  num_batches = len(data) // batch_size

  for epoch in range(EPOCH):
    np.random.shuffle(data)

    # Loop through the num batches
    for batch_number in range(num_batches):
      # Initialize x_batch and y_batch
      x_batch, y_batch = [], []

      for each_data in data[batch_size * batch_number : batch_size * (batch_number + 1)]:
        file_path = os.path.join(os.path.join(DATASET_DIR, each_data[1]), each_data[0])

        # Read image file
        img = cv2.imread(file_path)

        # Preprocess image
        frame = preprocess_image(img, img_height,img_weight)

        x_batch.append(frame)
        y_batch.append(CLASS_NAMES.index(each_data[1]))

      x_batch = np.array(x_batch)
      y_batch = np.array(y_batch)
      y_batch = tf.constant(y_batch)
      y_batch = tf.one_hot(y_batch, depth=NUM_CLASSES)

      yield x_batch, y_batch

In [48]:
print(f"Num GPUs available: {len(tf.config.list_physical_devices('GPU'))}")

Num GPUs available: 1


# Implement Model

## Implement Simple Model

In [49]:
def simple_model(train_data, test_data):
  img_height = 224
  img_weight = 224

  train_generator = data_generator(train_data, img_height=img_height, img_weight=img_weight)
  test_generator = data_generator(test_data, img_height=img_height, img_weight=img_weight)

  model = keras.Sequential()

  model.add(
      layers.Conv2D(
          64,
          kernel_size=(3, 3),
          strides=(2, 2),
          padding="same",
          activation="relu",
          input_shape=(img_height, img_weight, 3)
      )
  )

  model.add(
      layers.Conv2D(
          64,
          kernel_size=(3, 3),
          strides=(2, 2),
          padding="same",
          activation="relu"
      )
  )

  model.add(
      layers.MaxPooling2D(
          pool_size=(2, 2),
          strides=(2, 2)
      )
  )

  model.add(
      layers.Conv2D(
          128,
          kernel_size=(5, 5),
          strides=(2, 2),
          padding="same",
          activation="relu"
      )
  )

  model.add(
      layers.Conv2D(
          128,
          kernel_size=(5, 5),
          strides=(2, 2),
          padding="same",
          activation="relu"
      )
  )

  model.add(layers.Dropout(0.5))

  model.add(
      layers.Flatten()
  )

  model.add(
      layers.Dense(
          NUM_CLASSES,
          activation="softmax"
      )
  )
  # Compile model
  model.compile(
      optimizer="adam",
      loss="categorical_crossentropy",
      metrics=["accuracy"]
  )

  # Print model summary
  model.summary()

  history = model.fit(
      train_generator,
      steps_per_epoch=train_data_length // BATCH_SIZE,
      epochs=EPOCH,
      validation_data=test_generator,
      validation_steps=test_data_length // BATCH_SIZE
  )


## Implement Inception V3

In [50]:
def inception_v3_model(train_data, test_data, num_layers_to_unfreeze=1000, use_weight=False):
  train_generator = data_generator(train_data)
  test_generator = data_generator(test_data)

  base_model = keras.applications.InceptionV3(
      weights="imagenet",
      include_top=False,
      input_shape=(299, 299, 3)
  )

  model_path = f"{GDRIVE_DATASET_DIR}/model_weights/inception_v3.h5"
  if os.path.exists(model_path) and use_weight:
    base_model.load_weights(model_path)

  x = base_model.output
  x = layers.GlobalAveragePooling2D()(x)
  x = layers.Dense(1024, activation='relu')(x)
  predictions = layers.Dense(NUM_CLASSES, activation='softmax')(x)

  model = keras.models.Model(inputs=base_model.input, outputs=predictions)

  if num_layers_to_unfreeze < 1000:
    for layer in base_model.layers:
      layer.trainable = False

    for layer in model.layers[-num_layers_to_unfreeze:]:
      layer.trainable = True

  optimizer = keras.optimizers.SGD(
      learning_rate=0.001,
      momentum=0.9
  )

  model.compile(
      optimizer=optimizer,
      loss='categorical_crossentropy',
      metrics=['accuracy']
  )

  history = model.fit(
      train_generator,
      steps_per_epoch=train_data_length // BATCH_SIZE,
      epochs=EPOCH,
      validation_data=test_generator,
      validation_steps=test_data_length // BATCH_SIZE
  )

  model.save_weights(model_path)

  # Evaluate the model on the test dataset
  test_generator = data_generator(test_data)  # Reset the test generator to start from the beginning
  y_true = []
  y_pred = []

  start_time = datetime.now()
  for _ in range(test_data_length // BATCH_SIZE):
      X_batch, y_batch = next(test_generator)
      y_true.extend(np.argmax(y_batch, axis=1))
      y_pred.extend(np.argmax(model.predict(X_batch), axis=1))
  end_time = datetime.now()

  time_taken = (end_time - start_time).total_seconds()
  avg_time_taken_per_image = time_taken / ((test_data_length // BATCH_SIZE) * BATCH_SIZE)

  f1 = f1_score(y_true, y_pred, average='weighted')
  precision = precision_score(y_true, y_pred, average='weighted')
  recall = recall_score(y_true, y_pred, average='weighted')
  conf_matrix = confusion_matrix(y_true, y_pred)

  return history, f1, precision, recall, conf_matrix, avg_time_taken_per_image

## Implement ResNet V2

In [51]:
def res_net_v2_model(train_data, test_data, num_layers_to_unfreeze=1000, use_weight=False):
  train_generator = data_generator(train_data)
  test_generator = data_generator(test_data)

  base_model = keras.applications.ResNet50V2(
      weights="imagenet",
      include_top=False,
      input_shape=(299, 299, 3)
  )

  model_path = f"{GDRIVE_DATASET_DIR}/model_weights/resnet_v2.h5"
  if os.path.exists(model_path) and use_weight:
    base_model.load_weights(model_path)

  x = base_model.output
  x = layers.GlobalAveragePooling2D()(x)
  x = layers.Dense(1024, activation='relu')(x)
  predictions = layers.Dense(NUM_CLASSES, activation='softmax')(x)

  model = keras.models.Model(inputs=base_model.input, outputs=predictions)

  if num_layers_to_unfreeze < 1000:
    for layer in base_model.layers:
      layer.trainable = False

    for layer in model.layers[-num_layers_to_unfreeze:]:
      layer.trainable = True

  optimizer = keras.optimizers.SGD(
      learning_rate=0.001,
      momentum=0.9
  )

  model.compile(
      optimizer=optimizer,
      loss='categorical_crossentropy',
      metrics=['accuracy']
  )

  history = model.fit(
      train_generator,
      steps_per_epoch=train_data_length // BATCH_SIZE,
      epochs=EPOCH,
      validation_data=test_generator,
      validation_steps=test_data_length // BATCH_SIZE
  )

  model.save_weights(model_path)

  # Evaluate the model on the test dataset
  test_generator = data_generator(test_data)  # Reset the test generator to start from the beginning
  y_true = []
  y_pred = []

  start_time = datetime.now()
  for _ in range(test_data_length // BATCH_SIZE):
      X_batch, y_batch = next(test_generator)
      y_true.extend(np.argmax(y_batch, axis=1))
      y_pred.extend(np.argmax(model.predict(X_batch), axis=1))
  end_time = datetime.now()

  time_taken = (end_time - start_time).total_seconds()
  avg_time_taken_per_image = time_taken / ((test_data_length // BATCH_SIZE) * BATCH_SIZE)

  f1 = f1_score(y_true, y_pred, average='weighted')
  precision = precision_score(y_true, y_pred, average='weighted')
  recall = recall_score(y_true, y_pred, average='weighted')
  conf_matrix = confusion_matrix(y_true, y_pred)

  return history, f1, precision, recall, conf_matrix, avg_time_taken_per_image

## Implment EfficientNet Model

In [52]:
def efficient_net_model(train_data, test_data, num_layers_to_unfreeze=1000, use_weight=False):
  img_height=224
  img_weight=224

  train_generator = data_generator(train_data, img_height=img_height, img_weight=img_weight)
  test_generator = data_generator(test_data, img_height=img_height, img_weight=img_weight)

  base_model = keras.applications.EfficientNetB0(
      weights="imagenet",
      include_top=False,
      input_shape=(img_height, img_weight, 3)
  )

  model_path = f"{GDRIVE_DATASET_DIR}/model_weights/efficientnet.h5"
  if os.path.exists(model_path) and use_weight:
    base_model.load_weights(model_path)

  x = base_model.output
  x = layers.GlobalAveragePooling2D()(x)
  x = layers.Dense(1024, activation='relu')(x)
  predictions = layers.Dense(NUM_CLASSES, activation='softmax')(x)

  model = keras.models.Model(inputs=base_model.input, outputs=predictions)

  if num_layers_to_unfreeze < 1000:
    for layer in base_model.layers:
      layer.trainable = False

    for layer in model.layers[-num_layers_to_unfreeze:]:
      layer.trainable = True

  optimizer = keras.optimizers.SGD(
      learning_rate=0.001,
      momentum=0.9
  )

  model.compile(
      optimizer=optimizer,
      loss='categorical_crossentropy',
      metrics=['accuracy']
  )

  history = model.fit(
      train_generator,
      steps_per_epoch=train_data_length // BATCH_SIZE,
      epochs=EPOCH,
      validation_data=test_generator,
      validation_steps=test_data_length // BATCH_SIZE
  )

  model.save_weights(model_path)

  # Evaluate the model on the test dataset
  test_generator = data_generator(test_data, img_height=img_height, img_weight=img_weight)  # Reset the test generator to start from the beginning
  y_true = []
  y_pred = []

  start_time = datetime.now()
  for _ in range(test_data_length // BATCH_SIZE):
      X_batch, y_batch = next(test_generator)
      y_true.extend(np.argmax(y_batch, axis=1))
      y_pred.extend(np.argmax(model.predict(X_batch), axis=1))
  end_time = datetime.now()

  time_taken = (end_time - start_time).total_seconds()
  avg_time_taken_per_image = time_taken / ((test_data_length // BATCH_SIZE) * BATCH_SIZE)

  f1 = f1_score(y_true, y_pred, average='weighted')
  precision = precision_score(y_true, y_pred, average='weighted')
  recall = recall_score(y_true, y_pred, average='weighted')
  conf_matrix = confusion_matrix(y_true, y_pred)

  return history, f1, precision, recall, conf_matrix, avg_time_taken_per_image

## Implement VGG19 Model

In [57]:
def vgg_19_model(train_data, test_data, num_layers_to_unfreeze=1000, use_weight=False):
  img_height=224
  img_weight=224

  train_generator = data_generator(train_data, img_height=img_height, img_weight=img_weight)
  test_generator = data_generator(test_data, img_height=img_height, img_weight=img_weight)

  base_model = keras.applications.VGG19(
      weights="imagenet",
      include_top=False,
      input_shape=(img_height, img_weight, 3)
  )

  model_path = f"{GDRIVE_DATASET_DIR}/model_weights/vgg19.h5"
  if os.path.exists(model_path) and use_weight:
    base_model.load_weights(model_path)

  x = base_model.output
  x = layers.GlobalAveragePooling2D()(x)
  x = layers.Dense(1024, activation='relu')(x)
  predictions = layers.Dense(NUM_CLASSES, activation='softmax')(x)

  model = keras.models.Model(inputs=base_model.input, outputs=predictions)

  if num_layers_to_unfreeze < 1000:
    for layer in base_model.layers:
      layer.trainable = False

    for layer in model.layers[-num_layers_to_unfreeze:]:
      layer.trainable = True

  optimizer = keras.optimizers.SGD(
      learning_rate=0.001,
      momentum=0.9
  )

  model.compile(
      optimizer=optimizer,
      loss='categorical_crossentropy',
      metrics=['accuracy']
  )

  history = model.fit(
      train_generator,
      steps_per_epoch=train_data_length // BATCH_SIZE,
      epochs=EPOCH,
      validation_data=test_generator,
      validation_steps=test_data_length // BATCH_SIZE
  )

  model.save_weights(model_path)

  # Evaluate the model on the test dataset
  test_generator = data_generator(test_data, img_height=img_height, img_weight=img_weight)  # Reset the test generator to start from the beginning
  y_true = []
  y_pred = []

  start_time = datetime.now()
  for _ in range(test_data_length // BATCH_SIZE):
      X_batch, y_batch = next(test_generator)
      y_true.extend(np.argmax(y_batch, axis=1))
      y_pred.extend(np.argmax(model.predict(X_batch), axis=1))
  end_time = datetime.now()

  time_taken = (end_time - start_time).total_seconds()
  avg_time_taken_per_image = time_taken / ((test_data_length // BATCH_SIZE) * BATCH_SIZE)

  f1 = f1_score(y_true, y_pred, average='weighted')
  precision = precision_score(y_true, y_pred, average='weighted')
  recall = recall_score(y_true, y_pred, average='weighted')
  conf_matrix = confusion_matrix(y_true, y_pred)

  return history, f1, precision, recall, conf_matrix, avg_time_taken_per_image

# Run Model and Print all the Metrics

In [53]:
history, f1, precision, recall, conf_matrix, avg_time_taken_per_image = inception_v3_model(train_data, test_data, num_layers_to_unfreeze=50, use_weight=False)

print("F1 Score:", f1)
print("Precision:", precision)
print("Recall:", recall)
print("Confusion Matrix:")
print(conf_matrix)
print(f"Average time taken to classify each image: {avg_time_taken_per_image:f} seconds")


87910968/87910968 [==============================] - 3s 0us/step
Epoch 1/10
55/55 [==============================] - 47s 485ms/step - loss: 1.1876 - accuracy: 0.5545 - val_loss: 0.8275 - val_accuracy: 0.6974
Epoch 2/10
55/55 [==============================] - 25s 461ms/step - loss: 0.6397 - accuracy: 0.7832 - val_loss: 0.6115 - val_accuracy: 0.7784
Epoch 3/10
55/55 [==============================] - 28s 505ms/step - loss: 0.4876 - accuracy: 0.8358 - val_loss: 0.5036 - val_accuracy: 0.8146
Epoch 4/10
55/55 [==============================] - 28s 504ms/step - loss: 0.3961 - accuracy: 0.8676 - val_loss: 0.4488 - val_accuracy: 0.8345
Epoch 5/10
55/55 [==============================] - 28s 503ms/step - loss: 0.3330 - accuracy: 0.8929 - val_loss: 0.4138 - val_accuracy: 0.8494
Epoch 6/10
55/55 [==============================] - 27s 498ms/step - loss: 0.2836 - accuracy: 0.9116 - val_loss: 0.3836 - val_accuracy: 0.8558
Epoch 7/10
55/55 [==============================] - 26s 473ms/step - loss: 0.

In [54]:
history, f1, precision, recall, conf_matrix, avg_time_taken_per_image = res_net_v2_model(train_data, test_data, num_layers_to_unfreeze=50, use_weight=False)

print("F1 Score:", f1)
print("Precision:", precision)
print("Recall:", recall)
print("Confusion Matrix:")
print(conf_matrix)
print(f"Average time taken to classify each image: {avg_time_taken_per_image:f} seconds")


94668760/94668760 [==============================] - 3s 0us/step
Epoch 1/10
55/55 [==============================] - 51s 716ms/step - loss: 1.0505 - accuracy: 0.6128 - val_loss: 0.7311 - val_accuracy: 0.7358
Epoch 2/10
55/55 [==============================] - 36s 651ms/step - loss: 0.4283 - accuracy: 0.8727 - val_loss: 0.4479 - val_accuracy: 0.8430
Epoch 3/10
55/55 [==============================] - 38s 686ms/step - loss: 0.2395 - accuracy: 0.9423 - val_loss: 0.3630 - val_accuracy: 0.8700
Epoch 4/10
55/55 [==============================] - 37s 678ms/step - loss: 0.1474 - accuracy: 0.9753 - val_loss: 0.3410 - val_accuracy: 0.8807
Epoch 5/10
55/55 [==============================] - 37s 685ms/step - loss: 0.0949 - accuracy: 0.9884 - val_loss: 0.3138 - val_accuracy: 0.8899
Epoch 6/10
55/55 [==============================] - 36s 662ms/step - loss: 0.0562 - accuracy: 0.9960 - val_loss: 0.3220 - val_accuracy: 0.8935
Epoch 7/10
55/55 [==============================] - 36s 652ms/step - loss: 0.

In [55]:
history, f1, precision, recall, conf_matrix, avg_time_taken_per_image = efficient_net_model(train_data, test_data, num_layers_to_unfreeze=100, use_weight=False)

print("F1 Score:", f1)
print("Precision:", precision)
print("Recall:", recall)
print("Confusion Matrix:")
print(conf_matrix)
print(f"Average time taken to classify each image: {avg_time_taken_per_image:f} seconds")


16705208/16705208 [==============================] - 1s 0us/step
Epoch 1/10
55/55 [==============================] - 38s 470ms/step - loss: 1.5798 - accuracy: 0.2818 - val_loss: 1.6178 - val_accuracy: 0.2209
Epoch 2/10
55/55 [==============================] - 18s 334ms/step - loss: 1.5041 - accuracy: 0.3341 - val_loss: 1.6138 - val_accuracy: 0.1832
Epoch 3/10
55/55 [==============================] - 18s 322ms/step - loss: 1.4296 - accuracy: 0.3713 - val_loss: 1.6101 - val_accuracy: 0.1882
Epoch 4/10
55/55 [==============================] - 19s 356ms/step - loss: 1.3863 - accuracy: 0.3869 - val_loss: 1.6218 - val_accuracy: 0.1768
Epoch 5/10
55/55 [==============================] - 22s 400ms/step - loss: 1.3552 - accuracy: 0.3946 - val_loss: 1.6328 - val_accuracy: 0.1875
Epoch 6/10
55/55 [==============================] - 22s 397ms/step - loss: 1.3367 - accuracy: 0.3926 - val_loss: 1.6753 - val_accuracy: 0.1861
Epoch 7/10
55/55 [==============================] - 22s 398ms/step - loss: 1.

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [58]:
history, f1, precision, recall, conf_matrix, avg_time_taken_per_image = vgg_19_model(train_data, test_data, num_layers_to_unfreeze=50, use_weight=False)

print("F1 Score:", f1)
print("Precision:", precision)
print("Recall:", recall)
print("Confusion Matrix:")
print(conf_matrix)
print(f"Average time taken to classify each image: {avg_time_taken_per_image:f} seconds")


80134624/80134624 [==============================] - 3s 0us/step
Epoch 1/10
55/55 [==============================] - 90s 1s/step - loss: 1.2388 - accuracy: 0.4977 - val_loss: 0.9484 - val_accuracy: 0.6392
Epoch 2/10
55/55 [==============================] - 65s 1s/step - loss: 0.7846 - accuracy: 0.7028 - val_loss: 0.7266 - val_accuracy: 0.7280
Epoch 3/10
55/55 [==============================] - 65s 1s/step - loss: 0.6583 - accuracy: 0.7474 - val_loss: 0.5536 - val_accuracy: 0.7933
Epoch 4/10
55/55 [==============================] - 63s 1s/step - loss: 0.5475 - accuracy: 0.7895 - val_loss: 0.5068 - val_accuracy: 0.8054
Epoch 5/10
55/55 [==============================] - 62s 1s/step - loss: 0.4462 - accuracy: 0.8276 - val_loss: 0.4901 - val_accuracy: 0.8068
Epoch 6/10
55/55 [==============================] - 65s 1s/step - loss: 0.4021 - accuracy: 0.8483 - val_loss: 0.4756 - val_accuracy: 0.8217
Epoch 7/10
55/55 [==============================] - 62s 1s/step - loss: 0.3906 - accuracy: 0.84